# Notebook 2 — Feature Bags, Manifests and DataLoaders

## Purpose of this notebook

This notebook converts the dataset metadata prepared in Notebook 1 into model-ready feature manifests and PyTorch dataloaders for weakly supervised Multiple Instance Learning.

Notebook 1 creates the dataset splits and configuration handoff. This notebook then prepares the actual inputs used by the MIL models in later notebooks.

The main outputs of this notebook are:

- fixed-length `.npy` feature bags for UCF-Crime and XD-Violence;
- `train_manifest.csv`, `val_manifest.csv`, and `test_manifest.csv`;
- feature-dimension verification summaries;
- manifest summary files;
- bag normalisation statistics fitted on the training split only;
- PyTorch `Dataset` and `DataLoader` objects for MIL training.

## Pipeline overview

```text
Notebook 1 metadata and splits
        ↓
Load UCF and XD train/validation/test metadata
        ↓
Create or load feature bags
        ↓
Build train/validation/test manifests
        ↓
Verify feature dimensions
        ↓
Fit bag normalisation on training features only
        ↓
Create PyTorch datasets and dataloaders
        ↓
Pass loaders to Baseline MIL and Attention MIL notebooks
```

The final submitted run uses a feature-based spatiotemporal setup:

```text
UCF-Crime → R(2+1)D-18 feature bags
XD-Violence → RGB + Flow I3D fused feature bags
Final MIL input shape → [num_segments, 512]
```

This notebook does not train the final anomaly detection model. Its role is to ensure that every video is represented as a consistent temporal feature bag suitable for weakly supervised MIL training.


# Notebook 1 → Notebook 2 configuration handoff

This notebook starts by loading the session file created by Notebook 1:

```text
artifacts/task1_session.json
```

The session file stores the project paths, dataset mode, feature mode, number of temporal segments, CLIP settings, and feature-dimension configuration. Loading this file ensures that Notebook 2 uses the same settings as Notebook 1.

The bootstrap cell below should be run first when starting from a fresh kernel. It restores:

- project root paths;
- metadata and split directories;
- processed feature directories;
- dataset mode: UCF, XD, or both;
- feature mode: 2D, 3D, or fusion;
- number of temporal segments;
- input feature dimensions;
- CLIP semantic configuration.

This avoids redefining paths manually and keeps the multi-notebook pipeline reproducible.


In [1]:
# Bootstrap from Task 1 session — run first on a fresh kernel
import json
import os
import platform
from pathlib import Path

import torch

def _submission_root() -> Path:
    env = os.environ.get("VAD_SUBMISSION_ROOT")
    if env:
        return Path(env).resolve()
    cwd = Path.cwd().resolve()
    if cwd.name == "notebooks":
        return cwd.parent
    return cwd

SUBMISSION_ROOT = _submission_root()
SESSION_PATH = SUBMISSION_ROOT / "artifacts" / "task1_session.json"
if not SESSION_PATH.exists():
    raise FileNotFoundError(
        f"Missing {SESSION_PATH}. Run Task 1 to completion and execute its session-export cell, "
        "or set VAD_SUBMISSION_ROOT to your dev_submission folder."
    )

sess = json.loads(SESSION_PATH.read_text(encoding="utf-8"))

PROJECT_ROOT = Path(sess["project_root"])
PROCESSED_ROOT = sess["processed_root"]
DATA_ROOT = sess["data_root"]
XD_VIOLENCE_ROOT = sess["xd_violence_root"]
I3D_FEATURES_DIR = Path(sess["i3d_features_dir"])
METADATA_DIR = Path(sess["metadata_dir"])
SPLITS_DIR = Path(sess["splits_dir"])
FEATURES_DIR = Path(sess["features_dir"])
FEATURES_DIR_XD = Path(sess["features_dir_xd"])
FEATURES_3D_DIR = Path(sess["features_3d_dir"])
FEATURES_FUSION_DIR = Path(sess["features_fusion_dir"])
RESULTS_DIR = Path(sess["results_dir"])

USE_RUNS_DIR = bool(sess["use_runs_dir"])
RUN_TAG = sess["run_tag"]
CFG = sess["cfg"]
SEED = int(sess["seed"])
NUM_SEGMENTS = int(sess["num_segments"])
DATASET_MODE = sess["dataset_mode"]
USE_FUSION = bool(sess["use_fusion"])
USE_3D_FEATURES = bool(sess["use_3d_features"])

# Extra-credit switch from Notebook 1.
USE_CLIP_TEXT_EXTRA_CREDIT = bool(sess.get("use_clip_text_extra_credit", True))
ANOMALY_LABEL_SET = sess.get("anomaly_label_set", [
    "normal",
    "abuse",
    "arrest",
    "arson",
    "assault",
    "burglary",
    "explosion",
    "fighting",
    "road accident",
    "robbery",
    "shooting",
    "shoplifting",
    "stealing",
    "vandalism",
    "violence",
])

FEATURE_DIM = int(sess["feature_dim"])
FEATURE_DIM_3D = int(sess["feature_dim_3d"])
XD_FEATURE_DIM = int(sess["xd_feature_dim"])
XD_SEGMENTS_RAW = int(sess["xd_segments_raw"])
XD_STREAM_OUT_DIM = int(sess["xd_stream_out_dim"])
FRAMES_PER_SEGMENT = int(sess["frames_per_segment"])
RESIZE_HW = tuple(sess["resize_hw"])
CLIP_LEN = int(sess["clip_len"])
CLIP_STRIDE = int(sess["clip_stride"])
TRAIN_RATIO = float(sess["train_ratio"])
VAL_RATIO = float(sess["val_ratio"])
TEST_RATIO = float(sess["test_ratio"])
IN_COLAB = bool(sess["in_colab"])
PHASE_ACTIVE = int(sess.get("phase_active", 2))

_ = platform.system()
print("Loaded session:", SESSION_PATH)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_ROOT:", PROCESSED_ROOT)

FORCE_CPU = False

def select_torch_device(force_cpu: bool = False) -> torch.device:
    if force_cpu:
        print("TORCH_DEVICE: cpu (FORCE_CPU=True)")
        return torch.device("cpu")
    if not torch.cuda.is_available():
        print("TORCH_DEVICE: cpu (CUDA not available)")
        return torch.device("cpu")
    try:
        x = torch.randn(1, 3, 224, 224, device="cuda")
        w = torch.randn(64, 3, 7, 7, device="cuda")
        _ = torch.nn.functional.conv2d(x, w, padding=3)
        del _, w, x
        torch.cuda.empty_cache()
        print("TORCH_DEVICE: cuda (smoke test passed)")
        return torch.device("cuda")
    except Exception as e:
        print(f"CUDA reported but ops failed ({str(e)[:120]}...); using CPU")
        return torch.device("cpu")

TORCH_DEVICE = select_torch_device(FORCE_CPU)
device = TORCH_DEVICE

if USE_FUSION:
    MIL_INPUT_DIM = FEATURE_DIM + FEATURE_DIM_3D
elif USE_3D_FEATURES:
    MIL_INPUT_DIM = FEATURE_DIM_3D
else:
    MIL_INPUT_DIM = FEATURE_DIM

MIL_DUMMY_T = 32 if (USE_3D_FEATURES or USE_FUSION) else NUM_SEGMENTS

def ucf_feature_dir() -> Path:
    if USE_FUSION:
        return FEATURES_FUSION_DIR
    return FEATURES_3D_DIR if USE_3D_FEATURES else FEATURES_DIR

print("Bootstrap OK — continue with Section 6 onward.")

print("USE_CLIP_TEXT_EXTRA_CREDIT:", USE_CLIP_TEXT_EXTRA_CREDIT)
print("ANOMALY_LABEL_SET size:", len(ANOMALY_LABEL_SET))


Loaded session: /scratch/VAD/artifacts/task1_session.json
PROJECT_ROOT: /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217
PROCESSED_ROOT: /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217/processed
TORCH_DEVICE: cuda (smoke test passed)
Bootstrap OK — continue with Section 6 onward.
USE_CLIP_TEXT_EXTRA_CREDIT: True
ANOMALY_LABEL_SET size: 15


# 1. Load metadata splits from Notebook 1

This section loads the train, validation and test split files created in Notebook 1.

For UCF-Crime, the required files are:

```text
train_df.pkl
val_df.pkl
test_df.pkl
```

For XD-Violence, when `DATASET_MODE` is set to `xd` or `both`, the required files are:

```text
xd_train_df.pkl
xd_val_df.pkl
xd_test_df.pkl
```

These dataframes contain clip-level metadata and binary labels. They do not yet contain final model inputs. The next sections use these metadata rows to create or locate `.npy` feature bags.


In [2]:
# Load UCF train/val/test splits produced by Task 1 (required on a fresh kernel)
import pandas as pd

for _name in ("train_df", "val_df", "test_df"):
    _p = SPLITS_DIR / f"{_name}.pkl"
    if not _p.exists():
        raise FileNotFoundError(
            f"Missing {_p}. Run Task 1 through the split + session-export cells first."
        )

train_df = pd.read_pickle(SPLITS_DIR / "train_df.pkl")
val_df = pd.read_pickle(SPLITS_DIR / "val_df.pkl")
test_df = pd.read_pickle(SPLITS_DIR / "test_df.pkl")
print("Loaded Task 1 splits:", len(train_df), "train |", len(val_df), "val |", len(test_df), "test")

if DATASET_MODE in ("xd", "both"):
    for _name in ("xd_train_df", "xd_val_df", "xd_test_df"):
        _p = SPLITS_DIR / f"{_name}.pkl"
        if not _p.exists():
            raise FileNotFoundError(
                f"Missing {_p}. In Task 1, run the XD metadata/split cells, then Section 5 "
                "(writes xd_*.pkl next to UCF splits), then session export."
            )
    xd_train_df = pd.read_pickle(SPLITS_DIR / "xd_train_df.pkl")
    xd_val_df = pd.read_pickle(SPLITS_DIR / "xd_val_df.pkl")
    xd_test_df = pd.read_pickle(SPLITS_DIR / "xd_test_df.pkl")
    print(
        "Loaded XD splits:",
        len(xd_train_df),
        "train |",
        len(xd_val_df),
        "val |",
        len(xd_test_df),
        "test",
    )


Loaded Task 1 splits: 1420 train | 190 val | 290 test
Loaded XD splits: 15816 train | 3954 val | 4000 test


# 2. Segment sampling

This section defines helper functions for converting each clip into temporal segments.

The project uses a fixed number of temporal segments per video:

```text
NUM_SEGMENTS = 32
```

For UCF-Crime frame data, frames are sampled uniformly across the ordered frame list. The helper functions compute segment boundaries and load representative frames from each segment.

The main functions are:

```text
segment_boundaries(num_frames, num_segments)
load_segment_frames_from_list(frame_paths, start_idx, end_idx, ...)
```

The returned frame tensor has shape:

```text
[T, H, W, C]
```

where `T` is the number of sampled frames in the segment.

Uniform temporal sampling is used so that videos with different lengths can be represented using the same number of temporal segments. This is required for batching and MIL training.


In [3]:
# Define segment_boundaries() and load_segment_frames_from_list(); used to sample frames per segment for feature extraction.
import numpy as np
import cv2
from pathlib import Path
from typing import List, Tuple
from PIL import Image

def segment_boundaries(num_frames: int, num_segments: int) -> List[Tuple[int, int]]:
    if num_frames <= 0:
        return [(0, 0)] * num_segments
    edges = np.linspace(0, num_frames, num_segments + 1, dtype=int)
    return [(int(edges[i]), int(max(edges[i + 1], edges[i] + 1))) for i in range(num_segments)]

def sample_indices(start: int, end: int, num_samples: int) -> List[int]:
    if end <= start:
        return [start] * num_samples
    idxs = np.linspace(start, end - 1, num_samples)
    return [int(x) for x in idxs]

def load_frames(frame_paths: List[str]) -> np.ndarray:
    frames = []
    for p in frame_paths:
        img = Image.open(p).convert("RGB")
        frames.append(np.array(img))
    return np.stack(frames, axis=0)

def load_segment_frames_from_list(
    frame_paths: List[str],
    start_idx: int,
    end_idx: int,
    num_samples: int = 16,
    resize_hw: Tuple[int, int] = (224, 224),
) -> np.ndarray:
    n = len(frame_paths)
    if n == 0:
        return np.zeros((num_samples, resize_hw[0], resize_hw[1], 3), dtype=np.uint8)
    indices = sample_indices(start_idx, min(end_idx, n), num_samples)
    paths = [frame_paths[min(i, n - 1)] for i in indices]
    frames = load_frames(paths)
    out = np.stack([cv2.resize(f, (resize_hw[1], resize_hw[0])) for f in frames], axis=0)
    return out.astype(np.uint8)

bounds = segment_boundaries(100, NUM_SEGMENTS)
print("Segment boundaries (first 3):", bounds[:3])

Segment boundaries (first 3): [(0, 3), (3, 6), (6, 9)]


# 3. Feature extraction

This section defines the feature extraction process for UCF-Crime frame clips.

The notebook contains support for multiple feature modes:

1. **2D feature mode**  
   Uses a pretrained ResNet model to extract one feature vector per temporal segment.

2. **3D feature mode**  
   Uses a pretrained R(2+1)D-18 model to extract spatiotemporal clip features from ordered PNG frame sequences.

3. **Fusion mode**  
   Optionally combines 2D and 3D features by concatenating their feature vectors.

The final submitted run uses the 3D/spatiotemporal configuration:

```text
USE_3D_FEATURES = True
USE_FUSION = False
UCF feature extractor = R(2+1)D-18
UCF feature dimension = 512
```

The extracted feature bags are saved as `.npy` files. Each saved bag represents one video clip and has the shape:

```text
[num_segments, feature_dim]
```

For the final submitted configuration, the expected UCF feature shape is:

```text
[32, 512]
```

The feature extraction cells are mainly needed when feature files do not already exist. In the final pipeline, later sections build manifests from the saved `.npy` feature bags.


In [4]:
# ResNet feature extraction: load segment frames per clip, extract features, save one .npy per clip under FEATURES_DIR. Requires Section 6.
if PHASE_ACTIVE < 2:
    raise RuntimeError("Phase 1 only. Set PHASE_ACTIVE = 2 in Config to run pipeline.")
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from torchvision import models, transforms
from tqdm import tqdm

if "TORCH_DEVICE" not in globals():
    raise RuntimeError("Run Section 2 (Configuration) first — it defines TORCH_DEVICE after a CUDA smoke test.")
device = TORCH_DEVICE
print("Device:", device)

class ResNetFeatureExtractor(nn.Module):
    def __init__(self, backbone: str = "resnet50"):
        super().__init__()
        net = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.feature_net = nn.Sequential(*list(net.children())[:-1])
        self.feat_dim = 2048

    @torch.no_grad()
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        feats = self.feature_net(x).flatten(1)
        feats = feats.view(b, t, -1).mean(dim=1)
        return feats

feature_extractor = ResNetFeatureExtractor().to(device)
feature_extractor.eval()

img_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def extract_segment_feature(frame_paths: list, start_idx: int, end_idx: int) -> np.ndarray:
    clip = load_segment_frames_from_list(frame_paths, start_idx, end_idx, num_samples=FRAMES_PER_SEGMENT, resize_hw=RESIZE_HW)
    x = torch.stack([img_transform(f) for f in clip], dim=0).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = feature_extractor(x).cpu().numpy()[0]
    return feat

def extract_and_save_clip_features(df: pd.DataFrame, out_dir: Path, overwrite: bool = False) -> pd.DataFrame:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
        clip_id = row["clip_id"]
        frame_paths = row["frame_paths"]
        num_frames = row["num_frames"]
        label = int(row["label"])
        bag_path = out_dir / f"{clip_id}.npy"
        if bag_path.exists() and not overwrite:
            bag = np.load(bag_path)
        else:
            bounds = segment_boundaries(num_frames, NUM_SEGMENTS)
            bag = np.stack([extract_segment_feature(frame_paths, s, e) for s, e in bounds], axis=0)
            np.save(bag_path, bag)
        rows.append({"clip_id": clip_id, "label": label, "feature_path": str(bag_path), "num_segments": bag.shape[0], "feature_dim": bag.shape[1]})
    return pd.DataFrame(rows)

# Example: extract for a small subset first
# tiny = train_df.head(2)
# feature_df = extract_and_save_clip_features(tiny, FEATURES_DIR, overwrite=False)
# display(feature_df)



Device: cuda


In [5]:
# --- R(2+1)D clip extraction (in-notebook; no external .py). Used when USE_3D_FEATURES=True. ---
# Stacked ordered PNGs -> Kinetics-pretrained R(2+1)D-18 -> bag [num_clips, 512].
from pathlib import Path
from typing import List, Sequence

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision.models.video import R2Plus1D_18_Weights, r2plus1d_18


def sample_clip_paths(frame_paths: Sequence[str], clip_len: int, stride: int) -> List[List[str]]:
    """Overlapping clips of frame paths. Short videos: pad by repeating the last frame."""
    paths = list(frame_paths)
    n = len(paths)
    if n == 0:
        return []
    if n < clip_len:
        pad = [paths[-1]] * (clip_len - n)
        return [paths + pad]
    clips: List[List[str]] = []
    for start in range(0, n - clip_len + 1, stride):
        clips.append(paths[start : start + clip_len])
    return clips


def load_clip_tensor_tchw(frame_paths: Sequence[str]) -> torch.Tensor:
    """Load PNG paths to uint8 tensor [T, C, H, W] for R2Plus1D_18_Weights.transforms()."""
    frames = []
    for p in frame_paths:
        img = Image.open(p).convert("RGB")
        frames.append(np.asarray(img, dtype=np.uint8))
    vid = np.stack(frames, axis=0)
    return torch.from_numpy(vid).permute(0, 3, 1, 2).contiguous()


class R2Plus1DClipExtractor(nn.Module):
    """One clip -> 512-d vector (Kinetics-pretrained R(2+1)D-18, global pool)."""

    def __init__(self, device=None):
        super().__init__()
        weights = R2Plus1D_18_Weights.DEFAULT
        self.preprocess = weights.transforms()
        self.backbone = r2plus1d_18(weights=weights)
        self.output_dim = 512
        self._device = device or torch.device("cpu")
        self.to(self._device)

    @torch.no_grad()
    def forward(self, vid_tchw: torch.Tensor) -> torch.Tensor:
        vid_tchw = vid_tchw.cpu()
        x = self.preprocess(vid_tchw)
        if x.dim() == 4:
            x = x.unsqueeze(0)
        x = x.to(self._device)
        x = self.backbone.stem(x)
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)
        x = self.backbone.avgpool(x)
        return x.flatten(1).squeeze(0)


def extract_r2plus1d_bag(
    frame_paths: Sequence[str],
    clip_len: int,
    stride: int,
    extractor: R2Plus1DClipExtractor,
) -> np.ndarray:
    """Return array [num_clips, 512]."""
    rows = []
    for cp in sample_clip_paths(frame_paths, clip_len, stride):
        vid = load_clip_tensor_tchw(cp)
        feat = extractor(vid).float().cpu().numpy()
        rows.append(feat)
    if not rows:
        return np.zeros((0, extractor.output_dim), dtype=np.float32)
    return np.stack(rows, axis=0).astype(np.float32)


def extract_and_save_r2plus1d_for_df(df, out_dir: Path, device: torch.device, clip_len: int, stride: int, overwrite: bool = False):
    """One .npy bag per clip_id under out_dir (same role as extract_and_save_clip_features)."""
    import pandas as pd
    from tqdm import tqdm

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    extractor = R2Plus1DClipExtractor(device=device).eval()
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="R(2+1)D clips"):
        clip_id = row["clip_id"]
        frame_paths = list(row["frame_paths"])
        bag_path = out_dir / f"{clip_id}.npy"
        if bag_path.exists() and not overwrite:
            bag = np.load(bag_path)
        else:
            bag = extract_r2plus1d_bag(frame_paths, clip_len, stride, extractor)
            np.save(bag_path, bag)
        label = int(row["label"])
        rows.append(
            {
                "clip_id": clip_id,
                "label": label,
                "feature_path": str(bag_path),
                "num_segments": bag.shape[0],
                "feature_dim": bag.shape[1],
            }
        )
    return pd.DataFrame(rows)


# 3a. Optional debug feature extraction

This section is an optional debugging utility for extracting a small number of feature bags from both normal and anomalous classes.

It is useful for checking that:

- feature extraction runs correctly;
- both normal and anomalous classes are present;
- feature bags are saved to the expected directory;
- manifest and dataloader cells can run on a small sample.

This section is not used for the final reported results. The final experiments use full or precomputed feature bags rather than the small debug subset.

When `USE_3D_FEATURES = True`, the debug extraction writes R(2+1)D feature bags to:

```text
processed/features_3d/
```

When using 2D mode, it writes ResNet feature bags to:

```text
processed/features/
```


In [6]:
# ============================================================================
# OPTIONAL DEBUG FEATURE EXTRACTION — DISABLED FOR FINAL RUN
# ============================================================================
# This cell was useful during early debugging to create a tiny balanced subset.
# It must stay disabled in the final pipeline so it does not overwrite or confuse
# the full train/val/test manifests.

RUN_TINY_DEBUG_EXTRACTION = False

if not RUN_TINY_DEBUG_EXTRACTION:
    print("Tiny debug extraction is disabled for the final run. Skipping this cell.")
else:
    # Keep the original tiny-debug code below this line only for development.
    # Quick start: extract features for a small normal+anomaly subset so the train manifest has both classes. Re-run manifest and DataLoader after.
    normal_train = train_df[train_df["label"] == 0]
    anomaly_train = train_df[train_df["label"] == 1]
    if len(normal_train) == 0 or len(anomaly_train) == 0:
        print("train_df has only one class (normal:", len(normal_train), ", anomaly:", len(anomaly_train), "). Check Section 3/4: metadata must include NormalVideos and at least one anomaly class.")
    else:
        n_per_class = min(4, len(normal_train), len(anomaly_train))
        tiny_train = pd.concat([normal_train.head(n_per_class), anomaly_train.head(n_per_class)], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        normal_val = val_df[val_df["label"] == 0]
        anomaly_val = val_df[val_df["label"] == 1]
        tiny_val = pd.concat([normal_val.head(1), anomaly_val.head(1)], ignore_index=True)
        if USE_FUSION:
            import torch
            if "TORCH_DEVICE" not in globals():
                raise RuntimeError("Run Section 2 (Configuration) first — it sets TORCH_DEVICE.")
            _ = extract_and_save_clip_features(tiny_train, FEATURES_DIR, overwrite=False)
            _ = extract_and_save_clip_features(tiny_val, FEATURES_DIR, overwrite=False)
            _ = extract_and_save_r2plus1d_for_df(tiny_train, FEATURES_3D_DIR, TORCH_DEVICE, CLIP_LEN, CLIP_STRIDE, overwrite=False)
            _ = extract_and_save_r2plus1d_for_df(tiny_val, FEATURES_3D_DIR, TORCH_DEVICE, CLIP_LEN, CLIP_STRIDE, overwrite=False)
            print("ResNet ->", FEATURES_DIR, "| R(2+1)D ->", FEATURES_3D_DIR, "(run fusion precompute cell next)")
        elif USE_3D_FEATURES:
            import torch
            if "TORCH_DEVICE" not in globals():
                raise RuntimeError("Run Section 2 (Configuration) first — it sets TORCH_DEVICE.")
            _ = extract_and_save_r2plus1d_for_df(tiny_train, FEATURES_3D_DIR, TORCH_DEVICE, CLIP_LEN, CLIP_STRIDE, overwrite=False)
            _ = extract_and_save_r2plus1d_for_df(tiny_val, FEATURES_3D_DIR, TORCH_DEVICE, CLIP_LEN, CLIP_STRIDE, overwrite=False)
            print("R(2+1)D clip features ->", FEATURES_3D_DIR)
        else:
            _ = extract_and_save_clip_features(tiny_train, FEATURES_DIR, overwrite=False)
            _ = extract_and_save_clip_features(tiny_val, FEATURES_DIR, overwrite=False)
        print("Done. Re-run the manifest cell, then the DataLoader cell, then training.")


Tiny debug extraction is disabled for the final run. Skipping this cell.


# 3b. Full UCF feature extraction

This section extracts feature bags for the full UCF-Crime train, validation and test splits, or for a controlled capped subset if a limit is specified.

For the final 3D configuration, UCF clips are processed using:

```text
R(2+1)D-18 pretrained video model
```

Each clip is converted into a temporal feature bag and saved as:

```text
processed/features_3d/<clip_id>.npy
```

Expected final shape:

```text
[32, 512]
```

The cell can skip feature files that already exist, which avoids repeating expensive extraction after a restart.

The full feature extraction output is used to build the UCF manifest files in the next section.


In [7]:
# ============================================================================
# FULL UCF FEATURE EXTRACTION
# Extracts full (or capped) features for train / val / test
# In 3D mode:
# - uses R(2+1)D-18 clip extraction
# - saves outputs to FEATURES_3D_DIR
# None = use all clips
# Set 50 / 100 / 200 for faster debugging
# ============================================================================

FEATURE_EXTRACT_CAP_PER_CLASS = None  # None = full extraction

print("=" * 70)
print("FULL UCF FEATURE EXTRACTION")
print("=" * 70)

# ============================================================================
# 1. HELPER: OPTIONAL PER-CLASS CAPPING
# ============================================================================

def _subset_for_extraction(df: pd.DataFrame, cap_per_class: int):
    """Optionally cap each class for faster extraction."""
    if cap_per_class is None:
        return df

    parts = []
    for label in sorted(df["label"].unique()):
        sub = df[df["label"] == label].head(cap_per_class)
        parts.append(sub)

    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

# ============================================================================
# 2. BUILD EXTRACTION SUBSETS
# ============================================================================

train_to_extract = _subset_for_extraction(train_df, FEATURE_EXTRACT_CAP_PER_CLASS)
val_to_extract = _subset_for_extraction(val_df, FEATURE_EXTRACT_CAP_PER_CLASS)
test_to_extract = _subset_for_extraction(test_df, FEATURE_EXTRACT_CAP_PER_CLASS)

print("Extracting train:", len(train_to_extract), "| val:", len(val_to_extract), "| test:", len(test_to_extract))

# ============================================================================
# 3. RUN FEATURE EXTRACTION
# ============================================================================

if USE_FUSION:
    import torch

    if "TORCH_DEVICE" not in globals():
        raise RuntimeError("Run Section 2 (Configuration) first — it sets TORCH_DEVICE.")

    _ = extract_and_save_clip_features(train_to_extract, FEATURES_DIR, overwrite=False)
    _ = extract_and_save_clip_features(val_to_extract, FEATURES_DIR, overwrite=False)
    _ = extract_and_save_clip_features(test_to_extract, FEATURES_DIR, overwrite=False)

    _ = extract_and_save_r2plus1d_for_df(train_to_extract, FEATURES_3D_DIR, TORCH_DEVICE, CLIP_LEN, CLIP_STRIDE, overwrite=False)
    _ = extract_and_save_r2plus1d_for_df(val_to_extract, FEATURES_3D_DIR, TORCH_DEVICE, CLIP_LEN, CLIP_STRIDE, overwrite=False)
    _ = extract_and_save_r2plus1d_for_df(test_to_extract, FEATURES_3D_DIR, TORCH_DEVICE, CLIP_LEN, CLIP_STRIDE, overwrite=False)

    print("ResNet + R(2+1)D extracted.")
    print("Next: run fusion precompute cell ->", FEATURES_FUSION_DIR)

elif USE_3D_FEATURES:
    import torch

    if "TORCH_DEVICE" not in globals():
        raise RuntimeError("Run Section 2 (Configuration) first — it sets TORCH_DEVICE.")

    _ = extract_and_save_r2plus1d_for_df(train_to_extract, FEATURES_3D_DIR, TORCH_DEVICE, CLIP_LEN, CLIP_STRIDE, overwrite=False)
    _ = extract_and_save_r2plus1d_for_df(val_to_extract, FEATURES_3D_DIR, TORCH_DEVICE, CLIP_LEN, CLIP_STRIDE, overwrite=False)
    _ = extract_and_save_r2plus1d_for_df(test_to_extract, FEATURES_3D_DIR, TORCH_DEVICE, CLIP_LEN, CLIP_STRIDE, overwrite=False)

    print("R(2+1)D clip features extracted ->", FEATURES_3D_DIR)

else:
    _ = extract_and_save_clip_features(train_to_extract, FEATURES_DIR, overwrite=False)
    _ = extract_and_save_clip_features(val_to_extract, FEATURES_DIR, overwrite=False)
    _ = extract_and_save_clip_features(test_to_extract, FEATURES_DIR, overwrite=False)

    print("2D clip features extracted ->", FEATURES_DIR)

# ============================================================================
# 4. NEXT STEP
# ============================================================================

print("Done. Re-run the manifest cell, then the DataLoader cell, then training.")

FULL UCF FEATURE EXTRACTION
Extracting train: 1420 | val: 190 | test: 290


R(2+1)D clips: 100%|██████████| 290/290 [00:00<00:00, 12912.26it/s]

R(2+1)D clip features extracted -> /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217/processed/features_3d
Done. Re-run the manifest cell, then the DataLoader cell, then training.


# 3c. Optional early fusion precompute

This section is only used when:

```text
feature_mode = fusion
```

In fusion mode, the notebook combines 2D ResNet feature bags and 3D R(2+1)D feature bags by concatenating them along the feature dimension.

The fused feature shape is:

```text
[T, 2048 + 512] = [T, 2560]
```

where `T` is the aligned temporal length.

This section is not used in the final submitted run because the final configuration is:

```text
feature_mode = 3d
USE_FUSION = False
```

The code is kept as an experimental option, but the final models use 512-dimensional spatiotemporal feature bags instead of 2560-dimensional fused bags.


In [8]:
# Precompute fused bags: FEATURE_DIM (2048) + FEATURE_DIM_3D (512) -> 2560. Run after 2D+3D extraction when feature_mode=="fusion".
if not USE_FUSION:
    print("feature_mode is not fusion; skipping fusion precompute.")
else:
    import numpy as np
    from tqdm import tqdm

    _FUSED_DIM = int(FEATURE_DIM + FEATURE_DIM_3D)
    _mismatch_warn_cap = 5
    _mismatch_warned = [0]
    _min_ok_ratio = 0.9  # require almost all clips fused per split

    def _align_and_fuse(bag_2d: np.ndarray, bag_3d: np.ndarray, clip_id: str) -> np.ndarray:
        T2, D2 = bag_2d.shape
        T3, D3 = bag_3d.shape
        if D2 != FEATURE_DIM or D3 != FEATURE_DIM_3D:
            raise ValueError(f"{clip_id}: expected 2D dim {FEATURE_DIM} and 3D dim {FEATURE_DIM_3D}, got {D2} and {D3}")
        if abs(T2 - T3) > 5 and _mismatch_warned[0] < _mismatch_warn_cap:
            print(f"Warning: large T mismatch for {clip_id}: 2D={T2}, 3D={T3}")
            _mismatch_warned[0] += 1
        t = min(T2, T3)
        if t == 0:
            raise ValueError(f"{clip_id}: empty bag after alignment")
        fused = np.concatenate([bag_2d[:t], bag_3d[:t]], axis=1).astype(np.float32)
        assert fused.shape[1] == _FUSED_DIM, fused.shape
        return fused

    def build_fusion_bags_for_df(df, out_dir: Path, overwrite: bool = False) -> tuple:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        n_ok = 0
        n_skip = 0
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Fusion bags"):
            clip_id = row["clip_id"]
            fname = clip_id if str(clip_id).endswith(".npy") else f"{clip_id}.npy"
            path_2d = FEATURES_DIR / fname
            path_3d = FEATURES_3D_DIR / fname
            path_out = out_dir / fname
            if not path_2d.exists() or not path_3d.exists():
                print(f"Warning: missing 2D or 3D features for {clip_id}, skipping fusion")
                n_skip += 1
                continue
            if path_out.exists() and not overwrite:
                bag = np.load(path_out)
                if bag.shape[1] != _FUSED_DIM:
                    raise ValueError(f"{clip_id}: existing fusion has wrong dim {bag.shape}")
                n_ok += 1
                continue
            bag_2d = np.load(path_2d).astype(np.float32)
            bag_3d = np.load(path_3d).astype(np.float32)
            fused = _align_and_fuse(bag_2d, bag_3d, str(clip_id))
            np.save(path_out, fused)
            n_ok += 1
        return n_ok, n_skip

    for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        if len(df) == 0:
            continue
        n_ok, n_skip = build_fusion_bags_for_df(df, FEATURES_FUSION_DIR, overwrite=False)
        print(f"Fusion {name}: ok={n_ok} skipped={n_skip} (clips in split {len(df)})")
        need = max(1, int(_min_ok_ratio * len(df)))
        if n_ok < need:
            raise RuntimeError(
                f"Fusion: too few fused bags for {name} ({n_ok}/{len(df)}, need >= {need}). "
                "Run 2D and 3D extraction first, then re-run this cell."
            )
    print("Fusion precompute ->", FEATURES_FUSION_DIR)

feature_mode is not fusion; skipping fusion precompute.


# 4. Manifest export

This section builds CSV manifest files for MIL training and evaluation.

A manifest connects each clip to its saved feature bag and binary label. Each row contains:

```text
clip_id
label
feature_path
num_segments
```

The main manifest files are:

```text
train_manifest.csv
val_manifest.csv
test_manifest.csv
```

These files are saved under the metadata directory and are used by the PyTorch dataset class later in this notebook.

The manifest step is important because the training notebooks do not read raw frames directly. They read `.npy` feature bags through the manifest file paths.


In [9]:
# Define build_feature_manifest() and save_manifests(); columns: clip_id, label, feature_path, num_segments.
if PHASE_ACTIVE < 2:
    raise RuntimeError("Phase 1 only. Set PHASE_ACTIVE = 2 in Config to run pipeline.")
import pandas as pd
from pathlib import Path

import numpy as np

def build_feature_manifest(split_df: pd.DataFrame, features_dir: Path) -> pd.DataFrame:
    rows = []
    for _, row in split_df.iterrows():
        clip_id = row["clip_id"]
        label = int(row["label"])
        fname = clip_id if str(clip_id).endswith(".npy") else f"{clip_id}.npy"
        feature_path = features_dir / fname
        if not feature_path.exists():
            continue
        bag = np.load(feature_path)
        rows.append({"clip_id": clip_id, "label": label, "feature_path": str(feature_path), "num_segments": bag.shape[0]})
    return pd.DataFrame(rows)

def save_manifests(train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame, features_dir: Path, out_dir: Path) -> None:
    for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        manifest = build_feature_manifest(df, features_dir)
        path = out_dir / f"{name}_manifest.csv"
        manifest.to_csv(path, index=False)
        print(f"{name}_manifest.csv:", len(manifest), "rows")

# Run after feature extraction:
# save_manifests(train_df, val_df, test_df, FEATURES_DIR, METADATA_DIR)
print("Run the cell below to write manifest CSVs to METADATA_DIR.")

Run the cell below to write manifest CSVs to METADATA_DIR.


In [10]:
# Write train/val/test_manifest.csv to METADATA_DIR. Run after Section 5 (splits) and feature extraction.
_ucf_dir = ucf_feature_dir()
save_manifests(train_df, val_df, test_df, _ucf_dir, METADATA_DIR)

train_manifest.csv: 1420 rows
val_manifest.csv: 190 rows
test_manifest.csv: 290 rows


In [11]:
# Build train_manifest_df and val_manifest_df from UCF and/or XD according to DATASET_MODE. Required before DataLoader/training.
if PHASE_ACTIVE < 2:
    raise RuntimeError("Phase 1 only. Set PHASE_ACTIVE = 2 in Config to run pipeline.")
_ucf_feat_dir = ucf_feature_dir()
if DATASET_MODE == "ucf":
    train_manifest_df = build_feature_manifest(train_df, _ucf_feat_dir)
    val_manifest_df = build_feature_manifest(val_df, _ucf_feat_dir)
    print("Manifests from UCF. Train:", len(train_manifest_df), "| Val:", len(val_manifest_df))
elif DATASET_MODE == "xd":
    if "xd_train_df" not in globals() or "xd_val_df" not in globals():
        raise RuntimeError("Run Phase 2 (XD metadata) and Section 2b (XD feature bags) first.")
    train_manifest_df = build_feature_manifest(xd_train_df, FEATURES_DIR_XD)
    val_manifest_df = build_feature_manifest(xd_val_df, FEATURES_DIR_XD)
    print("Manifests from XD. Train:", len(train_manifest_df), "| Val:", len(val_manifest_df))
elif DATASET_MODE == "both":
    ucf_train = build_feature_manifest(train_df, _ucf_feat_dir)
    ucf_val = build_feature_manifest(val_df, _ucf_feat_dir)
    xd_train_m = build_feature_manifest(xd_train_df, FEATURES_DIR_XD)
    xd_val_m = build_feature_manifest(xd_val_df, FEATURES_DIR_XD)
    train_manifest_df = pd.concat([ucf_train, xd_train_m], ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    val_manifest_df = pd.concat([ucf_val, xd_val_m], ignore_index=True)
    print("Manifests from UCF+XD. Train:", len(train_manifest_df), "| Val:", len(val_manifest_df))
else:
    raise ValueError("DATASET_MODE must be 'ucf', 'xd', or 'both'.")
if DATASET_MODE == "ucf":
    test_manifest_df = build_feature_manifest(test_df, _ucf_feat_dir)
elif DATASET_MODE == "xd":
    test_manifest_df = build_feature_manifest(xd_test_df, FEATURES_DIR_XD)
else:
    test_manifest_df = pd.concat([build_feature_manifest(test_df, _ucf_feat_dir), build_feature_manifest(xd_test_df, FEATURES_DIR_XD)], ignore_index=True)
train_manifest_df.to_csv(METADATA_DIR / "train_manifest.csv", index=False)
val_manifest_df.to_csv(METADATA_DIR / "val_manifest.csv", index=False)
test_manifest_df.to_csv(METADATA_DIR / "test_manifest.csv", index=False)
if len(train_manifest_df) < 50 or len(val_manifest_df) < 20:
    print("Warning: very few clips in manifest. Validation AUC will be unreliable. Run more feature extraction (UCF and/or XD Section 2b), then re-run manifest and DataLoader.")


Manifests from UCF+XD. Train: 17236 | Val: 4144


# 5. Feature dimension verification

This section checks that all feature bags in the train, validation and test manifests have the same feature width.

This is especially important in the combined UCF+XD setting because the two datasets originate from different feature pipelines:

```text
UCF-Crime → R(2+1)D feature bags
XD-Violence → RGB + Flow I3D feature bags
```

For the final submitted run, both sources must align to:

```text
feature_dim = 512
```

The verification cell loads feature bags from the manifests and counts their feature dimensions. If mixed dimensions are found, the notebook raises an error before training. This prevents hidden mismatches from reaching the model.


In [12]:
# ============================================================================
# FEATURE DIMENSION VERIFICATION
# ============================================================================
# This confirms that all feature bags in each split have the same feature width.
# It is especially important for combined UCF+XD 3D mode, where UCF R(2+1)D
# and XD RGB/Flow I3D features must be aligned to the same dimension.

import numpy as np
import pandas as pd
from collections import Counter

def inspect_manifest_feature_dims(manifest_df: pd.DataFrame, split_name: str, max_examples: int = 3):
    dims = []
    examples = {}

    if manifest_df is None or len(manifest_df) == 0:
        print(f"{split_name}: empty manifest")
        return Counter(), {}

    for _, row in manifest_df.iterrows():
        arr = np.load(row.feature_path)
        if arr.ndim != 2:
            raise ValueError(
                f"{split_name}: expected 2D feature bag [T, D], got shape {arr.shape} "
                f"for {row.feature_path}"
            )

        d = int(arr.shape[1])
        dims.append(d)

        if d not in examples:
            examples[d] = []
        if len(examples[d]) < max_examples:
            examples[d].append(str(row.feature_path))

    dim_counts = Counter(dims)

    print("=" * 70)
    print(f"{split_name.upper()} FEATURE DIMENSION CHECK")
    print("=" * 70)
    print("Dimension counts:", dim_counts)

    for d, paths in examples.items():
        print(f"\nExamples for D={d}:")
        for p in paths:
            print("  ", p)

    if len(dim_counts) > 1:
        raise ValueError(
            f"{split_name}: mixed feature dimensions found: {dict(dim_counts)}. "
            "This will break bag normalization/training. Check UCF/XD feature alignment."
        )

    return dim_counts, examples


train_dim_counts, train_dim_examples = inspect_manifest_feature_dims(train_manifest_df, "train")
val_dim_counts, val_dim_examples = inspect_manifest_feature_dims(val_manifest_df, "val")
test_dim_counts, test_dim_examples = inspect_manifest_feature_dims(test_manifest_df, "test")

# Store the verified input feature dimension for later notebooks.
if len(train_dim_counts) > 0:
    VERIFIED_INPUT_FEATURE_DIM = int(next(iter(train_dim_counts.keys())))
else:
    VERIFIED_INPUT_FEATURE_DIM = None

print("\nVerified input feature dimension:", VERIFIED_INPUT_FEATURE_DIM)

# In 3D mode this should normally be 512 after UCF/XD alignment.
if USE_3D_FEATURES and VERIFIED_INPUT_FEATURE_DIM is not None:
    print(
        "3D/spatiotemporal mode verified:",
        f"input bags are {VERIFIED_INPUT_FEATURE_DIM}-dimensional."
    )

TRAIN FEATURE DIMENSION CHECK
Dimension counts: Counter({512: 17236})

Examples for D=512:
   /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217/processed/features_xd/Spectre.2015__#01-56-13_01-57-08_label_B2-G-B1__1.npy
   /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217/processed/features_xd/v=U6GNhmLnjWU__#1_label_A__1.npy
   /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217/processed/features_xd/Salt.2010__#01-23-40_01-25-00_label_A__0.npy
VAL FEATURE DIMENSION CHECK
Dimension counts: Counter({512: 4144})

Examples for D=512:
   /scratch/VAD/experiments/runs/vad_BEST_Final_Run_Zain_ClipWeight_0.05_seg32_both_3d_topkK4_lr1e-05_bs4_seed42_ep40_hc2f41927_20260529_174217/processed/features_3d/Normal_Videos750_x264.npy

# 6. Manifest and dataset summary export

This section saves a summary of the dataset and manifest sizes.

The summary records:

- UCF train/validation/test dataframe sizes;
- XD train/validation/test dataframe sizes;
- final combined train/validation/test manifest sizes actually used by MIL.

The output file is:

```text
manifest_summary.csv
```

This file is useful for reporting and evaluation because it distinguishes between the original split sizes and the final number of feature bags available to the model.


In [13]:
# ============================================================================
# MANIFEST / DATASET SUMMARY EXPORT
# ============================================================================
# This summary is used later in Notebook 5 to avoid misleading final counts.
# It records UCF split sizes, XD split sizes, and combined manifest sizes.

summary_rows = []

# UCF dataframe split counts
if "train_df" in globals():
    summary_rows.append({"source": "UCF dataframe", "split": "train", "count": int(len(train_df))})
if "val_df" in globals():
    summary_rows.append({"source": "UCF dataframe", "split": "val", "count": int(len(val_df))})
if "test_df" in globals():
    summary_rows.append({"source": "UCF dataframe", "split": "test", "count": int(len(test_df))})

# XD dataframe split counts
if "xd_train_df" in globals():
    summary_rows.append({"source": "XD dataframe", "split": "train", "count": int(len(xd_train_df))})
if "xd_val_df" in globals():
    summary_rows.append({"source": "XD dataframe", "split": "val", "count": int(len(xd_val_df))})
if "xd_test_df" in globals():
    summary_rows.append({"source": "XD dataframe", "split": "test", "count": int(len(xd_test_df))})

# Final manifest counts actually used by MIL
summary_rows.append({"source": "Combined MIL manifest", "split": "train", "count": int(len(train_manifest_df))})
summary_rows.append({"source": "Combined MIL manifest", "split": "val", "count": int(len(val_manifest_df))})
summary_rows.append({"source": "Combined MIL manifest", "split": "test", "count": int(len(test_manifest_df))})

manifest_summary_df = pd.DataFrame(summary_rows)

manifest_summary_path = METADATA_DIR / "manifest_summary.csv"
manifest_summary_df.to_csv(manifest_summary_path, index=False)

print("Saved manifest summary:", manifest_summary_path)
display(manifest_summary_df)

Saved manifest summary: /scratch/VAD/artifacts/metadata/manifest_summary.csv


,source,split,count
0,UCF dataframe,train,1420
1,UCF dataframe,val,190
2,UCF dataframe,test,290
3,XD dataframe,train,15816
4,XD dataframe,val,3954
5,XD dataframe,test,4000
6,Combined MIL manifest,train,17236
7,Combined MIL manifest,val,4144
8,Combined MIL manifest,test,4290


# 7. Feature dimension summary export

This section saves a compact summary of the final feature configuration.

The output file is:

```text
feature_dimension_summary.csv
```

It records:

- dataset mode;
- feature mode;
- whether 3D features are enabled;
- whether fusion is enabled;
- verified input feature dimension;
- configured MIL input dimension;
- number of temporal segments;
- top-k setting.

This makes the final experiment configuration easier to verify from saved metadata files, especially in Notebook 5 and the report.


In [14]:
# ============================================================================
# FEATURE DIMENSION SUMMARY EXPORT
# ============================================================================
# This file helps Notebook 5/report clearly distinguish:
# - actual input feature dimension
# - configured MIL input dimension
# - feature mode and dataset mode

feature_dimension_summary_df = pd.DataFrame([{
    "dataset_mode": DATASET_MODE,
    "feature_mode": CFG["feature_mode"],
    "use_3d_features": bool(USE_3D_FEATURES),
    "use_fusion": bool(USE_FUSION),
    "verified_input_feature_dim": VERIFIED_INPUT_FEATURE_DIM,
    "mil_input_dim_config": int(MIL_INPUT_DIM),
    "feature_dim_2d": int(FEATURE_DIM),
    "feature_dim_3d": int(FEATURE_DIM_3D),
    "xd_stream_out_dim": int(XD_STREAM_OUT_DIM),
    "num_segments": int(NUM_SEGMENTS),
    "topk_k": int(CFG["topk_k"]),
    "note": (
        "3D/spatiotemporal feature mode: UCF R(2+1)D and XD RGB/Flow I3D bags "
        "should be aligned to the verified input dimension."
        if USE_3D_FEATURES
        else "2D/fusion feature mode; see feature_mode and verified_input_feature_dim."
    )
}])

feature_dimension_summary_path = METADATA_DIR / "feature_dimension_summary.csv"
feature_dimension_summary_df.to_csv(feature_dimension_summary_path, index=False)

print("Saved feature dimension summary:", feature_dimension_summary_path)
display(feature_dimension_summary_df)

Saved feature dimension summary: /scratch/VAD/artifacts/metadata/feature_dimension_summary.csv


,dataset_mode,feature_mode,use_3d_features,use_fusion,verified_input_feature_dim,mil_input_dim_config,feature_dim_2d,feature_dim_3d,xd_stream_out_dim,num_segments,topk_k,note
0,both,3d,True,False,512,512,2048,512,256,32,4,3D/spatiotemporal feature mode: UCF R(2+1)D an...


# 8. Bag normalisation

This section fits feature normalisation statistics using the training feature bags only.

The code computes:

```text
mean
standard deviation
feature dimension
number of training clips
number of training segments
```

and saves them to:

```text
bag_norm_stats.npz
```

The same training statistics are later applied to validation and test feature bags.

Fitting normalisation on the training split only avoids data leakage. Validation and test data should not influence the feature scaling parameters used by the model.


In [15]:
# ============================================================================
# BAG NORMALIZATION
# ============================================================================
# Fit mean/std on train bags only. Val/test use the same train statistics.
# This version checks feature dimensions before concatenation to prevent
# hidden UCF/XD mismatches.

import numpy as np
from collections import Counter

_norm_path = METADATA_DIR / "bag_norm_stats.npz"

if len(train_manifest_df) > 0:
    segments = []
    dims = []

    for _, row in train_manifest_df.iterrows():
        bag = np.load(row.feature_path).astype(np.float32)

        if bag.ndim != 2:
            raise ValueError(f"Expected bag [T, D], got {bag.shape} for {row.feature_path}")

        segments.append(bag)
        dims.append(int(bag.shape[1]))

    dim_counts = Counter(dims)
    print("Bag norm feature dimension counts:", dim_counts)

    if len(dim_counts) > 1:
        raise ValueError(
            f"Cannot fit bag normalization with mixed feature dimensions: {dict(dim_counts)}. "
            "Run the feature dimension verification cell and fix feature alignment first."
        )

    all_segments = np.concatenate(segments, axis=0)

    _mean = all_segments.mean(axis=0).astype(np.float32)
    _std = (all_segments.std(axis=0) + 1e-6).astype(np.float32)

    np.savez(
        _norm_path,
        mean=_mean,
        std=_std,
        feature_dim=np.array([all_segments.shape[1]], dtype=np.int32),
        num_train_clips=np.array([len(train_manifest_df)], dtype=np.int32),
        num_train_segments=np.array([all_segments.shape[0]], dtype=np.int32),
    )

    print("Bag norm fitted:")
    print("  train clips:", len(train_manifest_df))
    print("  train segments:", all_segments.shape[0])
    print("  feature dim:", all_segments.shape[1])
    print("  saved to:", _norm_path)

else:
    print("Train manifest empty; skipping bag norm fit.")

Bag norm feature dimension counts: Counter({512: 17236})
Bag norm fitted:
  train clips: 17236
  train segments: 639909
  feature dim: 512
  saved to: /scratch/VAD/artifacts/metadata/bag_norm_stats.npz


# 9. PyTorch MIL feature dataset

This section defines the PyTorch dataset class used for MIL training.

`MILFeatureDataset` reads the manifest dataframe and loads each `.npy` feature bag from disk.

Each item returned by the dataset contains:

```text
features
label
length
```

where:

- `features` is the temporal feature bag;
- `label` is the binary video-level label;
- `length` is the number of temporal segments before padding.

If bag normalisation statistics are provided, the dataset applies the training-set mean and standard deviation to each feature bag.


In [16]:
# PyTorch Dataset: loads .npy feature bags; optional norm_stats (mean, std) applied (fit on train, same for val/test).
import torch
from torch.utils.data import Dataset
import numpy as np

class MILFeatureDataset(Dataset):
    def __init__(self, manifest_df, norm_stats=None):
        self.df = manifest_df.reset_index(drop=True)
        self.norm_stats = norm_stats  # (mean, std) each shape (D,) or None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        features = np.load(row.feature_path).astype(np.float32)
        if self.norm_stats is not None:
            mean, std = self.norm_stats
            features = (features - mean) / std
        T = features.shape[0]
        return torch.tensor(features, dtype=torch.float32), torch.tensor(row.label), torch.tensor(T, dtype=torch.long)


# 10. DataLoaders and balanced batch sampling

This section creates the training and validation dataloaders used by the MIL models.

The notebook defines:

```text
BalancedBatchSampler
collate_pad_bags
train_loader
val_loader
```

The balanced sampler creates batches with approximately equal numbers of normal and anomalous videos. This is important for the MIL ranking loss because training depends on comparing abnormal bags against normal bags.

The collate function pads variable-length feature bags to the maximum temporal length in each batch and returns:

```text
features
labels
lengths
```

The resulting dataloaders are passed to the Baseline MIL and Attention MIL training notebooks.


In [17]:
# Create train_loader and val_loader from MILFeatureDataset. Requires non-empty train_manifest_df and val_manifest_df.
if PHASE_ACTIVE < 2:
    raise RuntimeError("Phase 1 only. Set PHASE_ACTIVE = 2 in Config to run pipeline.")
from torch.utils.data import DataLoader, Sampler

class BalancedBatchSampler(Sampler):
    """Yield batch indices so each batch has (approx) equal normal and anomaly samples."""
    def __init__(self, labels, batch_size, seed=42):
        self.labels = np.asarray(labels)
        self.batch_size = batch_size
        self.seed = seed
        self.normal_idx = np.where(self.labels == 0)[0]
        self.anomaly_idx = np.where(self.labels == 1)[0]

    def __iter__(self):
        rng = np.random.default_rng(self.seed)
        n_half = self.batch_size // 2
        n_batches = (len(self.labels) + self.batch_size - 1) // self.batch_size
        for _ in range(n_batches):
            n_n = min(n_half, len(self.normal_idx))
            n_a = min(self.batch_size - n_n, len(self.anomaly_idx))
            if n_n == 0: n_n = min(1, len(self.normal_idx))
            if n_a == 0: n_a = min(1, len(self.anomaly_idx))
            idx_n = rng.choice(self.normal_idx, size=n_n, replace=len(self.normal_idx) < n_n)
            idx_a = rng.choice(self.anomaly_idx, size=n_a, replace=len(self.anomaly_idx) < n_a)
            yield list(np.concatenate([idx_n, idx_a]))

    def __len__(self):
        return (len(self.labels) + self.batch_size - 1) // self.batch_size

def collate_pad_bags(batch):
    """Pad variable-length feature bags to max T in batch; return (feats, labels, lengths)."""
    feats = torch.nn.utils.rnn.pad_sequence([b[0] for b in batch], batch_first=True, padding_value=0.0)
    labels = torch.stack([b[1] for b in batch])
    lengths = torch.stack([b[2] for b in batch])
    return feats, labels, lengths

if len(train_manifest_df) == 0 or len(val_manifest_df) == 0:
    raise ValueError(
        "Manifests are empty. Run Section 7 (feature extraction) first: "
        "extract_and_save_clip_features(train_df/val_df, FEATURES_DIR) to create .npy feature bags."
    )

_norm_path = METADATA_DIR / "bag_norm_stats.npz"
norm_stats = None
if _norm_path.exists():
    _d = np.load(_norm_path)
    norm_stats = (_d["mean"], _d["std"])
    print("Using bag normalization (fit on train).")

train_dataset = MILFeatureDataset(train_manifest_df, norm_stats=norm_stats)
val_dataset = MILFeatureDataset(val_manifest_df, norm_stats=norm_stats)

USE_BALANCED_SAMPLER = True  # Each batch has both classes when possible
batch_size = int(CFG["batch_size"]) if "CFG" in globals() else 8
if USE_BALANCED_SAMPLER and len(train_manifest_df["label"].unique()) > 1:
    bal_sampler = BalancedBatchSampler(train_manifest_df["label"].values, batch_size, seed=SEED)
    train_loader = DataLoader(train_dataset, batch_sampler=bal_sampler, collate_fn=collate_pad_bags)
else:
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_pad_bags)
val_loader = DataLoader(val_dataset, batch_size=batch_size, collate_fn=collate_pad_bags)

if USE_FUSION:
    _xb, _, _ = next(iter(train_loader))
    assert int(_xb.shape[-1]) == int(FEATURE_DIM + FEATURE_DIM_3D), _xb.shape
    print("Fusion batch shape:", tuple(_xb.shape), "feature_dim=", int(_xb.shape[-1]))

if len(test_manifest_df) > 0:
    test_dataset = MILFeatureDataset(test_manifest_df, norm_stats=norm_stats)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_pad_bags)
else:
    test_loader = None
    print("test_manifest is empty — test_loader not created (test-set eval skipped).")

    # ============================================================================
# DATALOADER SHAPE CHECK
# ============================================================================
# Confirms the tensor contract used by Notebooks 3–5:
# features: [B, T, D], labels: [B], lengths: [B]

_xb, _yb, _lb = next(iter(train_loader))

print("=" * 70)
print("DATALOADER CONTRACT CHECK")
print("=" * 70)
print("features:", tuple(_xb.shape), "| dtype:", _xb.dtype)
print("labels:", tuple(_yb.shape), "| labels in batch:", _yb.tolist())
print("lengths:", tuple(_lb.shape), "| min/max:", int(_lb.min()), int(_lb.max()))
print("feature dim from loader:", int(_xb.shape[-1]))
print("MIL_INPUT_DIM config:", int(MIL_INPUT_DIM))

if int(_xb.shape[-1]) != int(MIL_INPUT_DIM):
    raise ValueError(
        f"Loader feature dim ({int(_xb.shape[-1])}) != MIL_INPUT_DIM ({int(MIL_INPUT_DIM)}). "
        "Fix feature dimension alignment or config before training."
    )

print("Dataloader contract OK.")

Using bag normalization (fit on train).
DATALOADER CONTRACT CHECK
features: (4, 32, 512) | dtype: torch.float32
labels: (4,) | labels in batch: [0, 0, 1, 1]
lengths: (4,) | min/max: 32 32
feature dim from loader: 512
MIL_INPUT_DIM config: 512
Dataloader contract OK.
